# MediAgent AI — M1 Radiology Agent: DenseNet-121 Training
### Chest X-Ray Multi-Label Pathology Classification on NIH ChestX-ray14
**Project:** MediAgent AI (VIT FF No. 180, Review 1)  
**Hardware Target:** Kaggle GPU (NVIDIA T4 or P100)  
**Architecture:** DenseNet-121 (Multi-label Binary Cross-Entropy with Pos-Weight Calibration)

In [ ]:
# 1. Verify GPU Availability
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"SUCCESS: GPU is active -> {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("WARNING: GPU is NOT active. Please go to Notebook Settings -> Accelerator -> GPU T4")
    device = torch.device("cpu")

In [ ]:
# 2. Locate NIH ChestX-ray14 Dataset in Kaggle Input
import os
from pathlib import Path
import pandas as pd

KAGGLE_INPUT = Path("/kaggle/input")
csv_candidates = list(KAGGLE_INPUT.glob("**/Data_Entry_2017*.csv"))

if not csv_candidates:
    raise FileNotFoundError(
        "Could not find Data_Entry_2017.csv! Please click '+ Add Input' in the right sidebar "
        "and search for 'nih-chest-xrays' to attach the dataset."
    )

csv_path = csv_candidates[0]
print(f"Found metadata CSV at: {csv_path}")

# Index all image files across image subdirectories
image_paths = {}
for p in KAGGLE_INPUT.glob("**/*.png"):
    image_paths[p.name] = str(p)

print(f"Indexed {len(image_paths):,} image files.")

In [ ]:
# 3. Load & Process Dataset Labels
NIH_LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

df = pd.read_csv(csv_path)
# Filter to images that actually exist on disk
df = df[df["Image Index"].isin(image_paths)].copy()
df["full_path"] = df["Image Index"].map(image_paths)

# Multi-hot encode the 14 pathologies
for label in NIH_LABELS:
    df[label] = df["Finding Labels"].apply(lambda x: 1 if label in x.split("|") else 0)

print(f"Total available samples: {len(df):,}")
print("Pathology prevalence:")
print(df[NIH_LABELS].sum().sort_values(ascending=False))

In [ ]:
# 4. Patient-Level Stratified Train/Val Split (No Data Leakage)
from sklearn.model_selection import train_test_split

unique_patients = df["Patient ID"].unique()
train_patients, val_patients = train_test_split(unique_patients, test_size=0.15, random_state=42)

train_df = df[df["Patient ID"].isin(train_patients)].copy()
val_df = df[df["Patient ID"].isin(val_patients)].copy()

# For fast, high-quality demonstration on Kaggle, use a balanced 10,000 image subset if dataset is huge
SAMPLE_LIMIT = 10000
if len(train_df) > SAMPLE_LIMIT:
    train_df = train_df.sample(n=SAMPLE_LIMIT, random_state=42)
if len(val_df) > 2000:
    val_df = val_df.sample(n=2000, random_state=42)

print(f"Training set: {len(train_df)} images | Validation set: {len(val_df)} images")

In [ ]:
# 5. PyTorch Dataset and DataLoader with Data Augmentation
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class NIHDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.paths = self.df["full_path"].values
        self.labels = self.df[NIH_LABELS].values.astype("float32")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(NIHDataset(train_df, train_transform), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(NIHDataset(val_df, val_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# 6. DenseNet-121 Model & Loss with Positive Weighting
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

model = densenet121(weights=DenseNet121_Weights.DEFAULT)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, len(NIH_LABELS))
model = model.to(device)

# Calculate positive class weights to counter extreme class imbalance
pos_counts = torch.tensor(train_df[NIH_LABELS].sum().values, dtype=torch.float32)
neg_counts = len(train_df) - pos_counts
pos_weight = (neg_counts / torch.clamp(pos_counts, min=1.0)).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
# 7. Model Training Loop (5 Epochs with Mixed Precision)
from tqdm.auto import tqdm

EPOCHS = 5
history = {"train_loss": [], "val_loss": []}

print("Starting GPU Training on DenseNet-121...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)

    scheduler.step()
    epoch_train_loss = train_loss / len(train_loader.dataset)
    history["train_loss"].append(epoch_train_loss)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)

    epoch_val_loss = val_loss / len(val_loader.dataset)
    history["val_loss"].append(epoch_val_loss)
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

In [ ]:
# 8. Compute Per-Class ROC-AUC and Plot ROC Curves
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc="Evaluating Validation Set"):
        imgs = imgs.to(device)
        probs = torch.sigmoid(model(imgs))
        all_preds.append(probs.cpu().numpy())
        all_targets.append(labels.numpy())

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

auc_scores = {}
plt.figure(figsize=(10, 8))
for i, label in enumerate(NIH_LABELS):
    if len(np.unique(all_targets[:, i])) > 1:
        auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
        auc_scores[label] = auc
        fpr, tpr, _ = roc_curve(all_targets[:, i], all_preds[:, i])
        plt.plot(fpr, tpr, label=f"{label} (AUC = {auc:.3f})")

mean_auc = np.mean(list(auc_scores.values()))
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title(f"M1 DenseNet-121 ROC Curves on NIH ChestX-ray14 (Mean AUC = {mean_auc:.3f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("radiology_roc_auc_curves.png", dpi=300)
plt.show()

print(f"\nOVERALL MEAN ROC-AUC: {mean_auc:.4f}")

In [ ]:
# 9. Plot Training Loss vs Epoch
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), history["train_loss"], 'o-', label="Train Loss")
plt.plot(range(1, EPOCHS + 1), history["val_loss"], 's-', label="Val Loss")
plt.title("M1 DenseNet-121 Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Weighted BCE Loss")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("radiology_loss_curve.png", dpi=300)
plt.show()

In [ ]:
# 10. Export to ONNX Format for MediAgent AI Backend
model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "mediagent_radiology_densenet121.onnx"

# Export wrapper that applies sigmoid so output is directly probabilities
class ONNXModelWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
    def forward(self, x):
        return torch.sigmoid(self.base_model(x))

export_wrapper = ONNXModelWrapper(model)

torch.onnx.export(
    export_wrapper,
    dummy_input,
    onnx_path,
    input_names=["radiograph_input"],
    output_names=["pathology_probabilities"],
    dynamic_axes={
        "radiograph_input": {0: "batch_size"},
        "pathology_probabilities": {0: "batch_size"},
    },
    opset_version=14,
)

print(f"SUCCESS: Exported ONNX model to {onnx_path} (File size: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB)")
print("You can now download this file and place it into `.model_cache/` in your MediAgent project!")